In [ ]:
from pathlib import Path

import polars as pl
import geopandas as gpd

In [ ]:
dir = Path(r"Q:\Data\Observed\Transit\Muni\APC\2025")

In [ ]:
# just to test and make sure a few things are true first
schema_overrides = {"ext_trip_id": str}#, "route_alpha_sort": float, "direction_0": float}
df_03_15_raw = pl.scan_csv(dir / "raw/OrbCAD_dbo_apc_correlated_2025_03_15.csv",
schema_overrides=schema_overrides, null_values="NA"
)
df_04_26_raw = pl.scan_csv(dir / "raw/OrbCAD_dbo_apc_correlated_2025_04_26.csv",
schema_overrides=schema_overrides, null_values="NA"
)

In [ ]:
bus_stop_non_id_cols = ["bs_sname",	"bs_lname",	"longitude", "latitude"]
route_cols = ["route_id", "direction_code_id", "route_alpha_sort", "route_alpha", "route_alpha_txt", "route_id_txt", "route_name", "route_desc"]

In [ ]:
# confirm bs_id.x == bs_id.y for all entries for 04_26 (03_15 doesn't have this issue)
df_04_26_raw.select((pl.col("bs_id.x") == pl.col("bs_id.y")).all()).collect()

In [ ]:
# confirm all entries in each bus stop column have the same value for each bus stop ID
display(df_03_15_raw.group_by("bs_id").agg(
    (pl.col(bus_stop_non_id_cols).n_unique() ==  1)
).select(pl.col(bus_stop_non_id_cols).all()).collect())
display(df_04_26_raw.group_by("bs_id.x").agg(
    (pl.col(bus_stop_non_id_cols).n_unique() ==  1)
).select(pl.col(bus_stop_non_id_cols).all()).collect())

In [ ]:
# confirm index is just a rising list of 1, 2, ..., len(df) + 1
for df in (df_03_15_raw, df_04_26_raw):
    len = df.select(pl.len()).collect().item()
    print(df.select((pl.col("index") == (pl.int_range(len) +1)).all()).collect().item())

In [ ]:
df_04_26 = (
    df_04_26_raw
    # only the 2025_04_26 file has the duplicate bs_id.x and bs_id.y
    .drop("bs_id.x")
    .rename({"bs_id.y": "bs_id"})
    .select(df_03_15_raw.collect_schema().names())  # get columns; reorder for concat later
    )

df = pl.concat((df_03_15_raw, df_04_26)).drop("index").with_columns(
    open_date_time=pl.col("act_trip_start_time", "open_date_time").str.strptime(pl.Datetime, "%+").dt.convert_time_zone("America/Los_Angeles")
).filter(pl.col("open_date_time").dt.month().is_in([4, 5])
    )

In [ ]:
bus_stops = df.select("route_alpha", "bs_id", *bus_stop_non_id_cols).unique().sort("bs_id").collect()
bus_stops.write_parquet(dir / "parsed/bus_stops.parquet")

In [ ]:
gpd.GeoDataFrame(
    bus_stops.to_pandas(),
    geometry=gpd.points_from_xy(  # assume WGS84 CRS
        bus_stops["longitude"], bus_stops["latitude"], crs="EPSG:4326"
    ),
# HOTFIX named -geo.parquet to differentiate it from non-geo parquet
# otherwise reading the geoparquet into polars would throw error
).to_parquet(dir / "parsed/bus_stops-geo.parquet")

In [ ]:
# rev_seconds (revenue seconds):
# the duration from this open_date_time to the next open_date_time, incldues dwell_time

In [ ]:
df.with_columns(
    close_date_time=pl.col("open_date_time") + pl.duration(seconds=pl.col("dwell_time"))
).collect().write_parquet(dir / "parsed/OrbCAD_dbo_apc_correlated_2025_AprMay.parquet")